# Lab 01 — First Steps with Pretrained and Generative Models

**Session 01 · Introduction to Generative and Pretrained Models**
*Desarrollo y Evaluación de Modelos Propios · Especialización en IA Generativa y Desarrollo de Negocios · Universidad del Rosario · Rosario GSB*

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/01-intro-generative-models/lab.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/01-intro-generative-models/lab.ipynb)

In today's slides you met generative models, the three Transformer families and the adaptation spectrum.
In this lab you touch each idea with code: you train the smallest possible model by hand, rebuild it in Keras,
then run three pretrained models from the Hugging Face Hub and look inside the text-generation loop.

### Learning objectives
1. Find your way around a Kaggle notebook: settings, Internet, saving and downloading.
2. Use PyTorch tensors and autograd to fit a line with gradient descent.
3. Build the same model in Keras 3 and compare both frameworks.
4. Run three pretrained models: a sentiment classifier, a masked-word predictor and a small chat model.
5. Inspect tokens and see how `temperature`, `top_k`, `top_p` and `max_new_tokens` change generated text.

### Time plan (60 min)
| Part | Topic | Time |
|---|---|---|
| 1 | Kaggle tour · PyTorch tensors and autograd | 15' |
| 2 | The same model in Keras 3 | 10' |
| 3 | Hugging Face pipelines and a chat model | 20' |
| 4 | Tokens and generation parameters | 15' |

### ✅ Kaggle checklist
- [ ] Your phone number is verified on Kaggle (required for Internet). Guide: [configuracion-kaggle.md](https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/docs/configuracion-kaggle.md).
- [ ] **Settings → Internet → On** (the notebook downloads models from the Hugging Face Hub).
- [ ] **Settings → Accelerator → None** is enough today: everything runs on CPU. A GPU (T4 x2) only makes Parts 3–4 a little faster, so save your quota.
- [ ] Run the cells from top to bottom with **Shift + Enter**.

**Legend:** ✋ **Checkpoint** = answer in the Zoom chat · 🧪 **Try it** = optional mini-exercise with a hidden solution.

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # models and datasets come from the Hugging Face Hub
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "No internet connection: this notebook downloads models and datasets from Hugging Face.\n"
            "Kaggle: in the right-hand panel open Settings (Session options) and switch Internet on. "
            "The switch only appears once your account is phone-verified (kaggle.com/settings). "
            "Then run this cell again. Colab has internet by default. "
            "On your own computer: check your connection, or set HF_HUB_OFFLINE=1 if the models are already cached."
        ) from None

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---
ensure("transformers")
ensure("keras")

## Part 1 — A Kaggle tour and PyTorch basics (15')

### 1.1 A two-minute tour of Kaggle
- **Cells:** code cells run Python; markdown cells (like this one) are text. `Shift + Enter` runs a cell and moves to the next one.
- **Settings** (menu bar or right-hand panel): *Accelerator* (CPU/GPU) and *Internet*. Changing the accelerator **restarts the session**, so run everything again from the top.
- **Add-ons → Secrets:** where API keys live (we use it from Session 06; keys are never written in code).
- **Save Version:** keeps a snapshot of your notebook in your Kaggle account.
- **File → Download notebook:** downloads the `.ipynb` file. You will need it to submit today's challenge.
- Each click on **Open in Kaggle** creates a *new copy*. To continue the one you started, open it from **Your Work**.

> ✋ **Checkpoint:** paste in the Zoom chat the line printed by the setup cell (`Device: … | Kaggle: …`). Does it say `cpu` or `cuda`?

### 1.2 Tensors
A **tensor** is an n-dimensional array (like a NumPy array) that can live on a GPU and remember how it was computed.
Every model in this course — from a line to SmolLM2 — is a function of tensors.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2)
print("a =\n", a)
print("shape:", tuple(a.shape), "| dtype:", a.dtype, "| device:", a.device)
print("a + b =\n", a + b)
print("a @ b (matrix product) =\n", a @ b)
print("mean of a:", a.mean().item())

### 1.3 Autograd: derivatives for free
If a tensor has `requires_grad=True`, PyTorch records every operation on it. Calling `.backward()` then computes the
derivative of the result with respect to that tensor. This is how every neural network is trained.

Example: $y = x^2 + 2x$, so $\frac{dy}{dx} = 2x + 2$, which equals 8 at $x = 3$.

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2 * x
y.backward()                      # computes dy/dx and stores it in x.grad
print(f"y = {y.item():.1f} | dy/dx at x = 3: {x.grad.item():.1f} (expected 2·3 + 2 = 8)")

### 1.4 Fit $y = 3x + 2$ with gradient descent
We generate noisy points around the line $y = 3x + 2$ and ask a model $\hat{y} = w \cdot x + b$ to find $w$ and $b$.
Each step of **gradient descent**:
1. **Forward pass:** compute predictions $\hat{y}$.
2. **Loss:** mean squared error, $L = \frac{1}{N}\sum (\hat{y} - y)^2$.
3. **Backward pass:** `loss.backward()` computes $\partial L / \partial w$ and $\partial L / \partial b$.
4. **Update:** move each parameter a small step (the *learning rate*) against its gradient.
5. **Reset** the gradients, because PyTorch accumulates them.

Training a large language model follows exactly this loop, with billions of parameters instead of two.

In [ ]:
N_POINTS = 64 if FAST_DEV_RUN else 256
N_STEPS = 60 if FAST_DEV_RUN else 200
LEARNING_RATE = 0.1

torch.manual_seed(SEED)
X = torch.linspace(-1, 1, N_POINTS).unsqueeze(1)           # shape (N, 1)
Y = 3 * X + 2 + 0.1 * torch.randn_like(X)                   # the true line plus a little noise

w = torch.zeros(1, requires_grad=True)                       # parameters start at 0
b = torch.zeros(1, requires_grad=True)
losses = []
for step in range(N_STEPS):
    y_hat = w * X + b                                         # 1. forward pass
    loss = ((y_hat - Y) ** 2).mean()                          # 2. mean squared error
    loss.backward()                                           # 3. gradients dL/dw and dL/db
    with torch.no_grad():                                     # 4. update without recording it
        w -= LEARNING_RATE * w.grad
        b -= LEARNING_RATE * b.grad
        w.grad.zero_()                                        # 5. reset the gradients
        b.grad.zero_()
    losses.append(loss.item())
    if step % 20 == 0 or step == N_STEPS - 1:
        print(f"step {step:3d} | loss {loss.item():.4f} | w = {w.item():.3f} | b = {b.item():.3f}")

print(f"\nLearned: y = {w.item():.2f}x + {b.item():.2f}   (true line: y = 3x + 2)")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(losses)
axes[0].set_yscale("log")
axes[0].set(title="Training loss (MSE, log scale)", xlabel="step", ylabel="loss")
axes[1].scatter(X.numpy(), Y.numpy(), s=10, alpha=0.5, label="data")
axes[1].plot(X.numpy(), (w * X + b).detach().numpy(), color="C1", linewidth=3, label="learned line")
axes[1].set(title="Data and learned line", xlabel="x", ylabel="y")
axes[1].legend()
plt.tight_layout()
plt.show()

> ✋ **Checkpoint:** what would happen if we removed the two `.zero_()` lines? Answer in the chat in one sentence.

🧪 **Try it:** set `LEARNING_RATE = 1.5` and run the training cell again. Then try `LEARNING_RATE = 0.01`. What happens to the loss and to $w$ in each case?

<details><summary>Show a solution</summary>

- **Without `.zero_()`** the gradients of every step are *added* to the previous ones, so the updates keep growing and training becomes unstable.
- **`LEARNING_RATE = 1.5`:** the steps are too large, the parameters overshoot the minimum and the loss **explodes** (it grows every step instead of shrinking).
- **`LEARNING_RATE = 0.01`:** the loss decreases, but slowly; after 200 steps $w$ is still clearly below 3.
- The learning rate is the first *hyperparameter* you meet in the course. Session 02 shows how to read loss curves to choose it.

</details>

## Part 2 — The same model in Keras 3 (10')

**Keras** is a high-level API: you describe the model and Keras writes the training loop for you.
**Keras 3 is multi-backend**: the same code runs on JAX, TensorFlow or PyTorch. The backend is chosen with the
`KERAS_BACKEND` environment variable *before* importing Keras. On Kaggle the default backend is TensorFlow (already
installed there); on a computer without TensorFlow the cell below switches Keras to PyTorch. The model code is identical on both.

The model is three lines: **define** (one `Dense` layer with one input and one output is exactly $w \cdot x + b$),
**compile** (choose optimizer and loss) and **fit** (train).

In [ ]:
import importlib.util
if importlib.util.find_spec("tensorflow") is None:   # e.g. a local install with PyTorch only
    os.environ.setdefault("KERAS_BACKEND", "torch")
import keras

print("Keras", keras.__version__, "| backend:", keras.backend.backend())

X_np, Y_np = X.numpy(), Y.numpy()                  # reuse the data from Part 1
EPOCHS = 20 if FAST_DEV_RUN else 100

keras.utils.set_random_seed(SEED)
model_k = keras.Sequential([keras.Input(shape=(1,)), keras.layers.Dense(1)])        # 1. define
model_k.compile(optimizer=keras.optimizers.SGD(learning_rate=0.1), loss="mse")       # 2. compile
history = model_k.fit(X_np, Y_np, epochs=EPOCHS, batch_size=32, verbose=0)         # 3. fit

w_k, b_k = model_k.layers[0].get_weights()
print(f"Keras learned: y = {w_k[0, 0]:.2f}x + {b_k[0]:.2f} | final loss {history.history['loss'][-1]:.4f}")

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "model": ["True line", "PyTorch (hand-written loop)", "Keras 3 (model.fit)"],
    "w": [3.0, w.item(), float(w_k[0, 0])],
    "b": [2.0, b.item(), float(b_k[0])],
}).round(3)
comparison

**What `fit()` did for you:** the forward pass, the loss, the backward pass, the update and the gradient reset
from Part 1, repeated over **mini-batches** of 32 points (one pass over all the data is an **epoch**).
Keras starts from small random weights and updates after every batch, so its numbers differ slightly from PyTorch's.

> ✋ **Checkpoint:** which version would you show a new analyst on your team, and which one gives you more control? Why?

🧪 **Try it:** replace the optimizer with `keras.optimizers.Adam(learning_rate=0.1)` and train again. Is the result closer to $w = 3, b = 2$ after the same number of epochs?

<details><summary>Show a solution</summary>

```python
keras.utils.set_random_seed(SEED)
model_k = keras.Sequential([keras.Input(shape=(1,)), keras.layers.Dense(1)])
model_k.compile(optimizer=keras.optimizers.Adam(learning_rate=0.1), loss="mse")
history = model_k.fit(X_np, Y_np, epochs=EPOCHS, batch_size=32, verbose=0)
print(model_k.layers[0].get_weights())
```
Adam adapts the step size of each parameter, so it usually gets close to $w = 3, b = 2$ in fewer epochs than plain SGD.
Compare the final `loss` values printed by both runs rather than trusting one number.

</details>

## Part 3 — Pretrained models from the Hugging Face Hub (20')

The [Hugging Face Hub](https://huggingface.co/models) hosts models, datasets and demo apps. Every model has a
**model card** (what it was trained on, its limits, its licence). The `pipeline()` function bundles the three pieces you
need: a **tokenizer** (text → numbers), the **model** and the **post-processing** (numbers → labels or text).
The first run downloads the weights, so Internet must be on.

You will use one model from each family seen in the slides:

| Model | Family | Pretraining objective | Task today |
|---|---|---|---|
| `distilbert/distilbert-base-uncased-finetuned-sst-2-english` | Autoencoding (encoder) | Distilled from BERT (Sanh et al., 2019), then fine-tuned on movie-review sentences (SST-2) | Sentiment classification |
| `google-bert/bert-base-uncased` | Autoencoding (encoder) | Masked language modelling | Fill in a masked word |
| `HuggingFaceTB/SmolLM2-135M-Instruct` | Autoregressive (decoder) | Next-token prediction, then instruction tuning | Chat |

### 3.1 Sentiment classification

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                     device=DEVICE)
reviews = [
    "The new mobile app is fast and very easy to use.",
    "I waited 40 minutes on the phone and nobody solved my problem.",
    "The branch staff were polite, but the loan process was painfully slow.",
    "La aplicación nueva es rápida y muy fácil de usar.",   # Spanish: the model was fine-tuned on English only
]
for text, pred in zip(reviews, sentiment(reviews)):
    print(f"{pred['label']:<8} {pred['score']:.3f}   {text}")

Look closely at the last two reviews:
- The **mixed** review has to be squeezed into one of only two labels: the model has no "mixed" or "neutral" option.
- The **Spanish** review still gets a label and a score, even though the model was fine-tuned on English movie reviews.
  **A high score is not a guarantee of a correct answer.**

💼 *Business lens:* before trusting a pretrained model, check its model card: language, domain and label set must match your use case.

### 3.2 Fill in the blank: what BERT was pretrained to do
BERT is an **autoencoding** model: during pretraining some words were hidden with `[MASK]` and the model learned to
recover them using the context on **both sides**. It does not write text left to right.

In [ ]:
fill_mask = pipeline("fill-mask", model="google-bert/bert-base-uncased", device=DEVICE)
for pred in fill_mask("The customer called the bank to [MASK] her credit card.", top_k=5):
    print(f"{pred['score']:.3f}  {pred['token_str']}")

> ✋ **Checkpoint:** change the end of the sentence to `"... to [MASK] her credit card because it was stolen."`
> Do the predictions change? What does that tell you about the context BERT uses?

### 3.3 A small chat model: SmolLM2-135M-Instruct
SmolLM2 is an **autoregressive** model (Allal et al., 2025) with about 135 million parameters: about 1,300 times
fewer than GPT-3. The *Instruct* version was further trained to follow instructions in a chat format.

Chat models expect a conversation formatted with special tokens. The tokenizer's **chat template** builds that
format for us from a list of messages. Below we first print what the model actually receives.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).float().to(DEVICE)   # float32: newer versions load bfloat16
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"{MODEL_NAME}: {n_params / 1e6:.0f} M parameters")

messages = [{"role": "user", "content": "Write a two-sentence welcome message for new customers of a digital bank."}]
print("\nWhat the model actually reads:\n")
print(tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False))

In [ ]:
MAX_NEW_TOKENS = 16 if FAST_DEV_RUN else 80


def encode_chat(messages):
    """Apply the chat template and return model inputs (a dict of tensors on DEVICE)."""
    enc = tokenizer.apply_chat_template(messages, add_generation_prompt=True,
                                        return_tensors="pt", return_dict=True)
    if not hasattr(enc, "keys"):                     # some older versions return a bare tensor
        enc = {"input_ids": enc, "attention_mask": torch.ones_like(enc)}
    return {k: v.to(DEVICE) for k, v in enc.items()}


def chat(prompt, max_new_tokens=MAX_NEW_TOKENS, seed=SEED, **generation_kwargs):
    """Send one user message and return only the newly generated text."""
    inputs = encode_chat([{"role": "user", "content": prompt}])
    torch.manual_seed(seed)                          # same seed + same settings = same sample
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                pad_token_id=tokenizer.eos_token_id, **generation_kwargs)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print(chat("Write a two-sentence welcome message for new customers of a digital bank.",
           do_sample=True, temperature=0.7, top_p=0.9))

> ✋ **Checkpoint:** would you send this message to a real customer as it is? Name one thing you would check first.

🧪 **Try it:** ask a factual question, for example `chat("When was the Universidad del Rosario founded?", do_sample=False)`,
and compare the answer with a trusted source. What does this tell you about using a 135M-parameter model as a source of facts?

<details><summary>Show a solution</summary>

Small models store very little world knowledge and still answer fluently, so they often **hallucinate**: dates, names
and numbers can be invented. (The Colegio Mayor de Nuestra Señora del Rosario was founded in Bogotá in 1653; check what
the model said.) Always verify facts in a primary source. For factual answers in a business setting you
would ground the model in trusted documents (RAG) or use a stronger model, and evaluate it on your own questions (Session 05).

</details>

## Part 4 — Tokens and generation parameters (15')

### 4.1 A peek at tokens
Models do not read letters or words: they read **tokens**, pieces of words mapped to integer **ids**.
SmolLM2 uses a byte-level BPE tokenizer; the symbol `Ġ` marks a token that starts with a space.

In [ ]:
text = "Generative models predict the next token."
tokens = tokenizer.tokenize(text)
ids = tokenizer(text)["input_ids"]
print("tokens :", tokens)
print("ids    :", ids)
print("decoded:", tokenizer.decode(ids))
print(f"{len(text.split())} words -> {len(tokens)} tokens")

### 4.2 English vs Spanish: same meaning, different token counts
API providers charge **per token**, and context windows are measured in tokens. A tokenizer trained mostly on English
may need more tokens for the same message in Spanish. Let's measure it with three sentences and their translations.

In [ ]:
pairs = [
    ("Thank you for contacting us. Your request has been received.",
     "Gracias por contactarnos. Su solicitud ha sido recibida."),
    ("The loan application requires a copy of your identity document.",
     "La solicitud de crédito requiere una copia de su documento de identidad."),
    ("Our customer service team will answer within two business days.",
     "Nuestro equipo de servicio al cliente responderá en dos días hábiles."),
]
rows = []
for english, spanish in pairs:
    for language, sentence in (("English", english), ("Spanish", spanish)):
        n_words, n_tokens = len(sentence.split()), len(tokenizer.tokenize(sentence))
        rows.append({"language": language, "sentence": sentence, "words": n_words,
                     "tokens": n_tokens, "tokens per word": round(n_tokens / n_words, 2)})
token_table = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
display(token_table)

totals = token_table.groupby("language")[["words", "tokens"]].sum()
totals["tokens per word"] = (totals["tokens"] / totals["words"]).round(2)
totals

> ✋ **Checkpoint:** by what percentage does the Spanish version need more (or fewer) tokens *per word* with this tokenizer? What would that mean for a monthly API bill?

🧪 **Try it:** repeat the count with the tokenizer of a more multilingual model, `Qwen/Qwen2.5-0.5B-Instruct` (only the tokenizer is downloaded, a few MB).

<details><summary>Show a solution</summary>

```python
qwen_tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
for english, spanish in pairs:
    print(len(qwen_tok.tokenize(english)), len(qwen_tok.tokenize(spanish)))
```
Token counts depend on the tokenizer: a vocabulary trained on more multilingual text usually splits Spanish into
fewer pieces. When you compare prices between providers, compare them **per task** (tokens your real messages need),
not only per million tokens.

</details>

### 4.3 The real version of the slide simulator
In the slides you moved a temperature slider over a toy vocabulary. Here is the same computation on SmolLM2's real
vocabulary (49,152 tokens): we take the scores (*logits*) for the next token after a prompt, apply softmax with
three temperatures and show the ten most likely tokens.

In [ ]:
prompt_text = "Thank you for contacting us. Your request has been"
inputs = tokenizer(prompt_text, return_tensors="pt").to(DEVICE)
with torch.no_grad():
    logits = model(**inputs).logits[0, -1]          # one score per vocabulary token

top = torch.topk(logits, 10)
next_token_table = pd.DataFrame({"token": [repr(tokenizer.decode(int(i))) for i in top.indices]})
for T in (0.5, 1.0, 1.5):
    probs = torch.softmax(logits / T, dim=-1)       # softmax with temperature
    next_token_table[f"p (T={T})"] = probs[top.indices].cpu().numpy().round(3)
print(f"Vocabulary size: {logits.shape[0]:,} tokens")
next_token_table

Low temperature concentrates the probability on the first token; high temperature spreads it over the others
(and over the ~49,000 tokens not shown). This is exactly what `generate()` does at every step when `do_sample=True`.

### 4.4 Generation parameters side by side
| Parameter | What it does |
|---|---|
| `do_sample` | `False` = greedy decoding (always the most likely token); `True` = sample from the distribution |
| `temperature` | Divides the logits before softmax: below 1 sharpens, above 1 flattens |
| `top_k` | Samples only among the *k* most likely tokens (`0` switches it off) |
| `top_p` | Samples among the smallest set of tokens whose probabilities add up to *p* (nucleus sampling; Holtzman et al., 2020) |
| `max_new_tokens` | Maximum length of the answer, in tokens: the text is cut there even mid-sentence |

We use the same prompt and the same seed for every setting, so the only thing that changes is the decoding strategy.

In [ ]:
PROMPT = "Write a short slogan for a coffee brand from the Andes."
SETTINGS = [
    {"name": "greedy", "do_sample": False},
    {"name": "T=0.3", "do_sample": True, "temperature": 0.3, "top_k": 0},
    {"name": "T=0.7, top_p=0.9", "do_sample": True, "temperature": 0.7, "top_k": 0, "top_p": 0.9},
    {"name": "T=1.0, top_k=50", "do_sample": True, "temperature": 1.0, "top_k": 50},
    {"name": "T=1.5, no filter", "do_sample": True, "temperature": 1.5, "top_k": 0},
    {"name": "T=0.7, max 8 tokens", "do_sample": True, "temperature": 0.7, "top_k": 0, "max_new_tokens": 8},
]

results = []
for setting in SETTINGS:
    kwargs = {k: v for k, v in setting.items() if k != "name"}
    kwargs.setdefault("max_new_tokens", MAX_NEW_TOKENS)
    output = chat(PROMPT, seed=SEED, **kwargs)
    words = output.lower().split()
    results.append({"setting": setting["name"], "words": len(words),
                    "unique words %": round(100 * len(set(words)) / max(1, len(words))),
                    "output": output})
display(pd.DataFrame(results))

> ✋ **Checkpoint:** which setting would you choose for **(a)** a bank's regulatory FAQ and **(b)** brainstorming campaign slogans? Answer in the chat with the setting names.

🧪 **Try it:** keep the setting `T=1.0, top_k=50` and generate three answers with seeds 1, 2 and 3. How different are they? Now do the same with `greedy`.

<details><summary>Show a solution</summary>

```python
for s in (1, 2, 3):
    print("sampled:", chat(PROMPT, seed=s, do_sample=True, temperature=1.0, top_k=50))
for s in (1, 2, 3):
    print("greedy :", chat(PROMPT, seed=s, do_sample=False))
```
With sampling, each seed gives a different slogan; with greedy decoding the seed does not matter and the output is
always the same. This is why you must **sample several outputs** before judging a setting: that is precisely what
today's challenge asks you to measure.

</details>

## Wrap-up

**Key takeaways**
- Training = forward pass → loss → backward pass → update, repeated. PyTorch shows every step; Keras wraps them in `fit()`.
- Keras 3 runs the same model code on JAX, TensorFlow or PyTorch.
- `pipeline()` gives you a pretrained model in one line, but its model card (language, domain, labels) decides whether it fits your case.
- Encoders such as BERT fill in and classify; decoders such as SmolLM2 generate one token at a time.
- Tokens are the unit of cost and context; the same message can need a different number of tokens in Spanish and English.
- Temperature, `top_k` and `top_p` trade consistency for diversity; none of them makes the answer true.

**Next: the Session Challenge.** You will generate several samples per prompt and temperature with SmolLM2, measure
their diversity (distinct-1 and distinct-2), decide which temperature fits each prompt and place a business scenario on
the adaptation spectrum. Open `challenge.ipynb` from the slides or the session guide.

💾 **Keep your work:** *Save Version* in Kaggle, or *File → Download notebook*.